# 08 — GNN Ghost Artist Detection
**Day 7: Graph Attention Network (GAT) + GCN baseline**

Model: `GhostDetectorGAT` — learns which graph connections matter most for ghost detection.
Dataset: 65 nodes (14 ghost, 51 organic), 692 edges, 8 node features.
Source: Neo4j (3 real artists) + Kaggle (11 ghost candidates, 50 organic controls).

In [1]:
import sys, json
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve,
)
from src.models.dataset_builder import build_dataset, PROCESSED, FEATURE_NAMES
from src.models.gnn_detector import GhostDetectorGAT, GhostDetectorGCN, train_one_epoch, evaluate

print("torch:", torch.__version__)
import torch_geometric; print("torch_geometric:", torch_geometric.__version__)

torch: 2.10.0
torch_geometric: 2.7.0


## 1. Load Dataset

In [2]:
data = build_dataset(max_organic=50, val_frac=0.20, test_frac=0.20, random_seed=42)

meta_path = PROCESSED / "gnn_dataset_meta.json"
meta = json.loads(meta_path.read_text())

print(f"Nodes:        {data.num_nodes}")
print(f"Edges:        {data.num_edges}")
print(f"Features:     {data.num_node_features}  {FEATURE_NAMES}")
print(f"Ghost:        {data.y.sum().item()}")
print(f"Organic:      {(data.y == 0).sum().item()}")
print(f"Train nodes:  {data.train_mask.sum().item()}")
print(f"Val nodes:    {data.val_mask.sum().item()}")
print(f"Test nodes:   {data.test_mask.sum().item()}")
print(f"\nClass balance (train): ghost={data.y[data.train_mask].float().mean():.1%}")

2026-04-15 18:07:52.688 | INFO     | src.models.dataset_builder:build_dataset:252 - Total nodes: 65
2026-04-15 18:07:52.689 | INFO     | src.models.dataset_builder:build_dataset:273 - Ghost nodes: 14, Organic nodes: 51
2026-04-15 18:07:52.695 | INFO     | src.models.dataset_builder:build_dataset:285 - Edges: 692
2026-04-15 18:07:52.703 | INFO     | src.models.dataset_builder:build_dataset:320 - Dataset saved to /Users/trimbkjagtap/eda-for-music/data/processed/gnn_dataset.pt
2026-04-15 18:07:52.707 | INFO     | src.models.dataset_builder:build_dataset:340 - Metadata saved to /Users/trimbkjagtap/eda-for-music/data/processed/gnn_dataset_meta.json


Nodes:        65
Edges:        692
Features:     8  ['track_count', 'closure_rate', 'tracks_per_day', 'hhi', 'total_variance', 'mean_duration_ms', 'isrc_prefix_count', 'genre_count']
Ghost:        14
Organic:      51
Train nodes:  39
Val nodes:    13
Test nodes:   13

Class balance (train): ghost=12.8%


## 2. Train GAT and GCN Models

In [3]:
EPOCHS = 200
LR = 0.005
WEIGHT_DECAY = 5e-4
HIDDEN = 32
HEADS = 4
DROPOUT = 0.3

in_ch = data.num_node_features
gat = GhostDetectorGAT(in_ch, hidden_channels=HIDDEN, heads=HEADS, dropout=DROPOUT)
gcn = GhostDetectorGCN(in_ch, hidden_channels=HIDDEN, dropout=DROPOUT)

opt_gat = torch.optim.Adam(gat.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
opt_gcn = torch.optim.Adam(gcn.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
crit = torch.nn.BCELoss()

gat_train_losses, gcn_train_losses = [], []
gat_val_accs, gcn_val_accs = [], []

print(f"{'Epoch':>6} | {'GAT Loss':>9} | {'GAT Val Acc':>11} | {'GCN Loss':>9} | {'GCN Val Acc':>11}")
print("-" * 60)

for epoch in range(EPOCHS):
    gat_loss = train_one_epoch(gat, data, opt_gat, crit)
    gcn_loss = train_one_epoch(gcn, data, opt_gcn, crit)
    gat_train_losses.append(gat_loss)
    gcn_train_losses.append(gcn_loss)

    if epoch % 20 == 0 or epoch == EPOCHS - 1:
        gat_m = evaluate(gat, data, data.val_mask)
        gcn_m = evaluate(gcn, data, data.val_mask)
        gat_val_accs.append((epoch, gat_m["accuracy"]))
        gcn_val_accs.append((epoch, gcn_m["accuracy"]))
        print(f"{epoch:>6} | {gat_loss:>9.4f} | {gat_m['accuracy']:>11.4f} | {gcn_loss:>9.4f} | {gcn_m['accuracy']:>11.4f}")

print("\nTraining complete.")

 Epoch |  GAT Loss | GAT Val Acc |  GCN Loss | GCN Val Acc
------------------------------------------------------------
     0 |    0.7593 |      0.7692 |    0.8429 |      0.2308
    20 |    0.1001 |      1.0000 |    0.3718 |      1.0000
    40 |    0.0165 |      1.0000 |    0.1818 |      1.0000
    60 |    0.0107 |      1.0000 |    0.0849 |      1.0000
    80 |    0.0055 |      1.0000 |    0.0171 |      1.0000
   100 |    0.0099 |      1.0000 |    0.0271 |      1.0000
   120 |    0.0045 |      1.0000 |    0.0173 |      1.0000
   140 |    0.0015 |      1.0000 |    0.0105 |      1.0000
   160 |    0.0014 |      1.0000 |    0.0036 |      1.0000
   180 |    0.0024 |      1.0000 |    0.0037 |      1.0000
   199 |    0.0022 |      1.0000 |    0.0026 |      1.0000

Training complete.


## 3. Evaluation on Test Set

In [4]:
gat_test = evaluate(gat, data, data.test_mask)
gcn_test = evaluate(gcn, data, data.test_mask)

# Rule-based baseline: use pre-computed signal scores from ex6_signal_report_card.csv
# Map to labels using the known ground-truth (RWN, MRC, Calmo=ghost; NF=organic)
# We evaluate it on the 4 real artists as the "rule-based test set"
RULE_TRUE =  [1, 1, 1, 0]   # ghost, ghost, ghost, organic
RULE_SCORES = [0.3566, 0.3313, 0.2201, 0.0233]  # from ex6
RULE_PRED = [1 if s >= 0.40 else 0 for s in RULE_SCORES]
rule_acc  = sum(p == t for p, t in zip(RULE_PRED, RULE_TRUE)) / len(RULE_TRUE)
rule_prec = precision_score(RULE_TRUE, RULE_PRED, zero_division=0)
rule_rec  = recall_score(RULE_TRUE, RULE_PRED, zero_division=0)
rule_f1   = f1_score(RULE_TRUE, RULE_PRED, zero_division=0)

print("=" * 72)
print(f"{'Model':<22} | {'Acc':>6} | {'Prec':>6} | {'Recall':>6} | {'F1':>6} | {'AUC-ROC':>8}")
print("-" * 72)

def fmt(m, probs=None, true=None):
    auc = "-"
    if probs is not None and true is not None and len(set(true)) == 2:
        try:
            auc = f"{roc_auc_score(true, probs):.4f}"
        except:
            pass
    return (f"{m['accuracy']:.4f}", f"{m['precision']:.4f}" if not str(m['precision']) == 'nan' else "N/A",
            f"{m['recall']:.4f}" if not str(m['recall']) == 'nan' else "N/A",
            f"{m['f1']:.4f}" if not str(m['f1']) == 'nan' else "N/A", auc)

gat_row = fmt(gat_test, gat_test["probs"], gat_test["true"])
gcn_row = fmt(gcn_test, gcn_test["probs"], gcn_test["true"])
rule_row = (f"{rule_acc:.4f}", f"{rule_prec:.4f}", f"{rule_rec:.4f}", f"{rule_f1:.4f}", "-")

print(f"{'GAT (ours)':<22} | {'  |  '.join(gat_row)}")
print(f"{'GCN (baseline)':<22} | {'  |  '.join(gcn_row)}")
print(f"{'Rule-based (verdict.py)':<22} | {'  |  '.join(rule_row)}")
print("=" * 72)

print("\nGAT Confusion Matrix (test set):")
cm = confusion_matrix(gat_test["true"], gat_test["pred"])
print(cm)
print(f"  TN={cm[0,0]}  FP={cm[0,1]}\n  FN={cm[1,0]}  TP={cm[1,1]}")

Model                  |    Acc |   Prec | Recall |     F1 |  AUC-ROC
------------------------------------------------------------------------
GAT (ours)             | 1.0000  |  1.0000  |  1.0000  |  1.0000  |  1.0000
GCN (baseline)         | 1.0000  |  1.0000  |  1.0000  |  1.0000  |  1.0000
Rule-based (verdict.py) | 0.2500  |  0.0000  |  0.0000  |  0.0000  |  -

GAT Confusion Matrix (test set):
[[7 0]
 [0 6]]
  TN=7  FP=0
  FN=0  TP=6


## 4. Feature Importance (Permutation)

In [5]:
@torch.no_grad()
def permutation_importance(model, data, mask, n_repeats=10, seed=42):
    """Shuffle each feature column, measure accuracy drop."""
    rng = np.random.default_rng(seed)
    base_m = evaluate(model, data, mask)
    base_acc = base_m["accuracy"]

    drops = {}
    for fi, fname in enumerate(FEATURE_NAMES):
        accs = []
        for _ in range(n_repeats):
            x_perm = data.x.clone()
            perm = rng.permutation(data.num_nodes)
            x_perm[:, fi] = x_perm[perm, fi]
            # Temporarily swap x
            orig_x = data.x
            data.x = x_perm
            m = evaluate(model, data, mask)
            data.x = orig_x
            accs.append(m["accuracy"])
        mean_perm_acc = float(np.mean(accs))
        drops[fname] = base_acc - mean_perm_acc

    return drops

feat_importance = permutation_importance(gat, data, data.test_mask)
fi_df = pd.DataFrame(list(feat_importance.items()), columns=["Feature", "Accuracy Drop"])
fi_df = fi_df.sort_values("Accuracy Drop", ascending=False).reset_index(drop=True)
print("Feature importance (accuracy drop when permuted):")
print(fi_df.to_string(index=False))

Feature importance (accuracy drop when permuted):
          Feature  Accuracy Drop
      track_count            0.0
     closure_rate            0.0
   tracks_per_day            0.0
              hhi            0.0
   total_variance            0.0
 mean_duration_ms            0.0
isrc_prefix_count            0.0
      genre_count            0.0


## 5. Attention Weights (GAT Interpretability)

In [6]:
gat.eval()
with torch.no_grad():
    probs, (edge_idx_out, alpha) = gat(data.x, data.edge_index, return_attention=True)

# alpha shape: [E, heads]  — average over heads
alpha_mean = alpha.mean(dim=-1).cpu().numpy()
src_nodes = edge_idx_out[0].cpu().numpy()
dst_nodes = edge_idx_out[1].cpu().numpy()
node_labels_arr = data.y.cpu().numpy()
node_names = meta["node_names"]

# For each ghost artist (from Neo4j), show top attention edges
ghost_neo4j = ["Relaxing White Noise", "Meditation Relax Club", "Calmo", "Nils Frahm"]
for artist in ghost_neo4j:
    if artist not in node_names:
        continue
    node_idx = node_names.index(artist)
    # Outgoing edges from this node
    out_mask = src_nodes == node_idx
    out_edges = np.where(out_mask)[0]
    if len(out_edges) == 0:
        print(f"\n{artist}: no outgoing edges")
        continue
    out_alpha = alpha_mean[out_mask]
    out_dst = dst_nodes[out_mask]
    top5_idx = np.argsort(out_alpha)[::-1][:5]
    label_str = "GHOST" if node_labels_arr[node_idx] == 1 else "ORGANIC"
    print(f"\n{artist} ({label_str}, node {node_idx}) — top attention edges:")
    for rank, ei in enumerate(top5_idx):
        dst = out_dst[ei]
        dst_name = node_names[dst] if dst < len(node_names) else f"node_{dst}"
        dst_label = "ghost" if node_labels_arr[dst] == 1 else "organic"
        print(f"  [{rank+1}] \u2192 {dst_name[:40]} ({dst_label}): \u03b1={out_alpha[ei]:.4f}")


Relaxing White Noise (GHOST, node 0) — top attention edges:
  [1] → Burna Boy;Ed Sheeran (ghost): α=0.1576
  [2] → Relaxing White Noise (ghost): α=0.1576
  [3] → Nessa Barrett (ghost): α=0.1576
  [4] → Madeline The Person (ghost): α=0.1576
  [5] → Tiësto;Charli XCX (ghost): α=0.1576

Meditation Relax Club (GHOST, node 1) — top attention edges:
  [1] → Relaxing White Noise (ghost): α=0.1115
  [2] → Nessa Barrett (ghost): α=0.1115
  [3] → Madeline The Person (ghost): α=0.1115
  [4] → Tiësto;Charli XCX (ghost): α=0.1115
  [5] → Tiësto;Ava Max (ghost): α=0.1115

Calmo (GHOST, node 2) — top attention edges:
  [1] → Calmo (ghost): α=0.0462
  [2] → Relaxing White Noise (ghost): α=0.0451
  [3] → Madeline The Person (ghost): α=0.0451
  [4] → Tiësto;Charli XCX (ghost): α=0.0451
  [5] → Neon Trees (ghost): α=0.0451

Nils Frahm (ORGANIC, node 3) — top attention edges:
  [1] → Nils Frahm (organic): α=0.1658
  [2] → Só Pra Contrariar (organic): α=0.0979
  [3] → Matheus & Kauan (organic): α=0.0973
 

## 6. Figure 7: GNN Performance

In [7]:
FIGURES_DIR = ROOT / "paper" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

fig = plt.figure(figsize=(16, 12), facecolor="#0d0d1a")
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.35)

DARK_BG  = "#0d0d1a"
PANEL_BG = "#1a1a2e"
COLOR_GAT = "#a78bfa"
COLOR_GCN = "#38bdf8"
COLOR_RULE = "#fb923c"
TEXT_COL  = "#e2e8f0"
GRID_COL  = "#2a2a4a"

def panel_ax(ax):
    ax.set_facecolor(PANEL_BG)
    ax.tick_params(colors=TEXT_COL, labelsize=9)
    for spine in ax.spines.values():
        spine.set_color(GRID_COL)
    ax.xaxis.label.set_color(TEXT_COL)
    ax.yaxis.label.set_color(TEXT_COL)
    ax.title.set_color(TEXT_COL)

# ── Top-left: Training loss curves ───────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
panel_ax(ax1)
epochs_range = range(1, EPOCHS + 1)
ax1.plot(epochs_range, gat_train_losses, color=COLOR_GAT, lw=1.5, label="GAT")
ax1.plot(epochs_range, gcn_train_losses, color=COLOR_GCN, lw=1.5, ls="--", label="GCN")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("BCE Loss")
ax1.set_title("Training Loss Curves")
ax1.legend(facecolor=PANEL_BG, labelcolor=TEXT_COL)
ax1.yaxis.grid(True, color=GRID_COL, lw=0.5)

# ── Top-right: ROC curves ────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
panel_ax(ax2)

def try_roc(probs, true, label, color, ax):
    if len(set(true)) < 2:
        ax.text(0.5, 0.5, f"{label}: only one class\nin test set",
                ha='center', va='center', color=TEXT_COL, transform=ax.transAxes)
        return
    fpr, tpr, _ = roc_curve(true, probs)
    auc = roc_auc_score(true, probs)
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{label} (AUC={auc:.3f})")

try_roc(gat_test["probs"], gat_test["true"], "GAT", COLOR_GAT, ax2)
try_roc(gcn_test["probs"], gcn_test["true"], "GCN", COLOR_GCN, ax2)
ax2.plot([0,1],[0,1], ":", color="#64748b", lw=1)
ax2.set_xlabel("False Positive Rate")
ax2.set_ylabel("True Positive Rate")
ax2.set_title("ROC Curves (Test Set)")
ax2.legend(facecolor=PANEL_BG, labelcolor=TEXT_COL, fontsize=9)
ax2.yaxis.grid(True, color=GRID_COL, lw=0.5)

# ── Bottom-left: Confusion matrix (GAT) ──────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
panel_ax(ax3)
cm = confusion_matrix(gat_test["true"], gat_test["pred"])
im = ax3.imshow(cm, cmap="RdPu", vmin=0)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax3.text(j, i, str(cm[i, j]), ha='center', va='center',
                 color='white', fontsize=14, fontweight='bold')
ax3.set_xticks([0, 1]); ax3.set_yticks([0, 1])
ax3.set_xticklabels(["Organic (0)", "Ghost (1)"], color=TEXT_COL)
ax3.set_yticklabels(["Organic (0)", "Ghost (1)"], color=TEXT_COL)
ax3.set_xlabel("Predicted")
ax3.set_ylabel("True")
ax3.set_title("GAT Confusion Matrix (Test Set)")
plt.colorbar(im, ax=ax3)

# ── Bottom-right: Feature importance ────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
panel_ax(ax4)
fi_sorted = fi_df.sort_values("Accuracy Drop")
colors_bar = [COLOR_GAT if d > 0 else COLOR_RULE for d in fi_sorted["Accuracy Drop"]]
ax4.barh(fi_sorted["Feature"], fi_sorted["Accuracy Drop"], color=colors_bar)
ax4.axvline(0, color=GRID_COL, lw=1)
ax4.set_xlabel("Accuracy Drop (higher = more important)")
ax4.set_title("Feature Importance (Permutation)")
ax4.xaxis.grid(True, color=GRID_COL, lw=0.5)

fig.suptitle(
    "Figure 7: GNN Ghost Artist Detection Performance\n"
    "GAT vs GCN vs Rule-based \u00b7 EDA for Music \u00b7 INFO 7390",
    color=TEXT_COL, fontsize=14, fontweight="bold", y=1.01,
)

out_png = FIGURES_DIR / "fig7_gnn_performance.png"
fig.savefig(out_png, dpi=300, bbox_inches="tight", facecolor=DARK_BG)
plt.close(fig)
print(f"Saved: {out_png}  ({out_png.stat().st_size // 1024} KB)")

Saved: /Users/trimbkjagtap/eda-for-music/paper/figures/fig7_gnn_performance.png  (452 KB)


## 7. Save Models

In [8]:
gat_path = PROCESSED / "gat_model.pt"
gcn_path = PROCESSED / "gcn_model.pt"

torch.save(gat.state_dict(), gat_path)
torch.save(gcn.state_dict(), gcn_path)
print(f"GAT model saved: {gat_path}")
print(f"GCN model saved: {gcn_path}")

# Save training summary
summary = {
    "gat": {
        "test_accuracy": gat_test["accuracy"],
        "test_precision": float(gat_test["precision"]) if not (str(gat_test["precision"]) == "nan") else None,
        "test_recall": float(gat_test["recall"]) if not (str(gat_test["recall"]) == "nan") else None,
        "test_f1": float(gat_test["f1"]) if not (str(gat_test["f1"]) == "nan") else None,
    },
    "gcn": {
        "test_accuracy": gcn_test["accuracy"],
        "test_precision": float(gcn_test["precision"]) if not (str(gcn_test["precision"]) == "nan") else None,
        "test_recall": float(gcn_test["recall"]) if not (str(gcn_test["recall"]) == "nan") else None,
        "test_f1": float(gcn_test["f1"]) if not (str(gcn_test["f1"]) == "nan") else None,
    },
    "rule_based": {
        "test_accuracy": rule_acc,
        "test_precision": float(rule_prec),
        "test_recall": float(rule_rec),
        "test_f1": float(rule_f1),
    },
    "top_features": fi_df["Feature"].tolist()[:3],
    "feature_importance": dict(zip(fi_df["Feature"], fi_df["Accuracy Drop"].round(4).tolist())),
    "dataset": {
        "num_nodes": data.num_nodes,
        "num_edges": data.num_edges,
        "num_ghost": int(data.y.sum()),
        "num_organic": int((data.y == 0).sum()),
    },
}
summary_path = PROCESSED / "gnn_training_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(f"\nTraining summary saved: {summary_path}")
print(json.dumps(summary, indent=2))

GAT model saved: /Users/trimbkjagtap/eda-for-music/data/processed/gat_model.pt
GCN model saved: /Users/trimbkjagtap/eda-for-music/data/processed/gcn_model.pt

Training summary saved: /Users/trimbkjagtap/eda-for-music/data/processed/gnn_training_summary.json
{
  "gat": {
    "test_accuracy": 1.0,
    "test_precision": 1.0,
    "test_recall": 1.0,
    "test_f1": 1.0
  },
  "gcn": {
    "test_accuracy": 1.0,
    "test_precision": 1.0,
    "test_recall": 1.0,
    "test_f1": 1.0
  },
  "rule_based": {
    "test_accuracy": 0.25,
    "test_precision": 0.0,
    "test_recall": 0.0,
    "test_f1": 0.0
  },
  "top_features": [
    "track_count",
    "closure_rate",
    "tracks_per_day"
  ],
  "feature_importance": {
    "track_count": 0.0,
    "closure_rate": 0.0,
    "tracks_per_day": 0.0,
    "hhi": 0.0,
    "total_variance": 0.0,
    "mean_duration_ms": 0.0,
    "isrc_prefix_count": 0.0,
    "genre_count": 0.0
  },
  "dataset": {
    "num_nodes": 65,
    "num_edges": 692,
    "num_ghost": 14,
